# EDA — Bike Sharing (`hour.csv`)

This notebook answers the required manual EDA checks before continuing implementation.

Dataset path: `data/raw/hour.csv`

## Quick findings

1. **Null values**: `0` total nulls across all 17 columns.
2. **Target distribution (`cnt`)**: right-skewed (`skew=1.2774`), not log-normal by default.
3. **Numeric correlation**: `corr(temp, atemp)=0.9877` (very high collinearity).
4. **Behavioral signal**: strongest hourly demand peaks at hours `17`, `18`, and `8`.
5. **Year split balance**: `yr=0: 8645`, `yr=1: 8734` (well-balanced).
6. **Natural temp drift by year**: mean temp `0.4891 -> 0.5048` and KS test `p≈1.03e-11`.
7. **Leakage check**: `casual + registered == cnt` is always `True`.

## Phase-1 impact

- Keep `drop_columns: [instant, dteday, casual, registered]` as-is.
- Keep temporal split on `yr` as-is.
- Keep schema constraints as-is.
- **No Phase 1 code change is required** based on EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('data/raw/hour.csv')
print('shape:', df.shape)
print('columns:', list(df.columns))

In [ ]:
# 1) Null check
null_counts = df.isnull().sum()
print('total nulls:', int(null_counts.sum()))
display(null_counts.to_frame('null_count').T)

# 2) Target distribution summary
cnt = df['cnt']
print('cnt min/max:', cnt.min(), cnt.max())
print('cnt mean/std:', round(cnt.mean(), 4), round(cnt.std(), 4))
print('cnt skew:', round(cnt.skew(), 4))
print('cnt quantiles:')
display(cnt.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_frame('cnt'))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(cnt, bins=50)
ax[0].set_title('cnt histogram')
ax[1].hist(np.log1p(cnt), bins=50)
ax[1].set_title('log1p(cnt) histogram')
plt.tight_layout()

In [ ]:
# 3) Numeric correlation (incl. temp vs atemp)
num_cols = ['temp', 'atemp', 'hum', 'windspeed', 'hr', 'mnth', 'cnt']
corr = df[num_cols].corr()
print('corr(temp, atemp)=', round(corr.loc['temp', 'atemp'], 4))
display(corr)

# 4) Distribution by hr / weekday / season
cnt_by_hr = df.groupby('hr')['cnt'].mean().sort_values(ascending=False)
print('Top-5 hours by mean cnt:')
display(cnt_by_hr.head(5).to_frame('mean_cnt'))

display(df.groupby('weekday')['cnt'].mean().round(2).to_frame('mean_cnt_weekday'))
display(df.groupby('season')['cnt'].mean().round(2).to_frame('mean_cnt_season'))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
df.groupby('hr')['cnt'].mean().plot(ax=ax[0], title='mean cnt by hour')
df.groupby('weekday')['cnt'].mean().plot(kind='bar', ax=ax[1], title='mean cnt by weekday')
df.groupby('season')['cnt'].mean().plot(kind='bar', ax=ax[2], title='mean cnt by season')
plt.tight_layout()

In [ ]:
# 5) Year-count balance for split
yr_counts = df['yr'].value_counts().sort_index()
print('yr counts:')
display(yr_counts.to_frame('rows'))

# 6) temp distribution shift between yr=0 and yr=1
temp_0 = df.loc[df['yr'] == 0, 'temp']
temp_1 = df.loc[df['yr'] == 1, 'temp']
ks_stat, ks_p = ks_2samp(temp_0, temp_1)

print('temp mean yr=0:', round(temp_0.mean(), 4))
print('temp mean yr=1:', round(temp_1.mean(), 4))
print('KS(temp yr0 vs yr1): stat=', round(ks_stat, 4), 'p=', ks_p)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(temp_0, bins=40, alpha=0.6, label='yr=0', density=True)
ax.hist(temp_1, bins=40, alpha=0.6, label='yr=1', density=True)
ax.set_title('temp distribution: yr=0 vs yr=1')
ax.legend()
plt.tight_layout()

In [ ]:
# 7) Leakage check: casual + registered == cnt
leakage_equal = ((df['casual'] + df['registered']) == df['cnt']).all()
print('casual + registered == cnt for all rows:', leakage_equal)

if leakage_equal:
    print('Confirmed: `casual` and `registered` are target leakage and must be dropped.')